In [7]:
from emeraldprocessing.tem import variance_averaging as va
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def plotData(raw_dat,
             stack_dat,
             stack_std,
             ave_dat,
             ave_std,
             figsize=(10, 5),
            ):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ind = []
    for ii, xn in enumerate(raw_dat):
        if ii==0:
            ax.plot(np.ones(len(xn)) * (ii), xn, c='grey', marker='o', linestyle='None', fillstyle='full', markeredgewidth=0.0, alpha=0.1, label='Raw data values, by group')
        else:
            ax.plot(np.ones(len(xn)) * (ii), xn, c='grey', marker='o', linestyle='None', fillstyle='full', markeredgewidth=0.0, alpha=0.1, label=None)
        ind.append(ii)
    ax.plot(np.ones(len(np.concatenate(raw_dat))) * ii/2, np.concatenate(raw_dat), c='lightcoral', marker='o', linestyle='None', fillstyle='full', markeredgewidth=0.0, alpha=0.1, label='Raw data values')
    ax.errorbar(ind, stack_dat, stack_std, c='k', fmt='o-', capsize=5, elinewidth=3, capthick=3, label='Group ave')
    ax.errorbar(ii/2, ave_dat, ave_std, c='r', fmt='o-', capsize=5, elinewidth=3, capthick=3, label='total ave')
    ax.legend()
    if ax.get_xlim()[1]<10:
        ax.set_xlim([-1, 10])
    else:
        ax.set_xlim([-1, ax.get_xlim()[1]+1])

# First just try two sets #############################

In [ ]:
sample_len = 50

In [ ]:
raw_0 = np.random.normal(loc=1, scale=2, size=sample_len)
raw_1 = np.random.normal(loc=2, scale=1, size=sample_len)

raw_0_ave = np.average(raw_0)
raw_0_std = np.std(raw_0)

raw_1_ave = np.average(raw_1)
raw_1_std = np.std(raw_1)

# print(f"raw_0 = {raw_0}\n")
# print(f"raw_1 = {raw_1}\n")

print(f"raw_0 ={raw_0_ave: 1.4f} ±{raw_0_std: 1.4f}")
print(f"raw_1 ={raw_1_ave: 1.4f} ±{raw_1_std: 1.4f}\n")

In [ ]:
raw_groups =[raw_0, raw_1]
raw_ave = [raw_0_ave, raw_1_ave]
raw_std = [raw_0_std, raw_1_std]

In [ ]:
mu_actual = np.average(np.concatenate((raw_0, raw_1)))
var_actual = np.var(np.concatenate((raw_0, raw_1)))
std_actual = np.std(np.concatenate((raw_0, raw_1)))

#print(f"x_average ={mu_actual: 1.4f} ±{var_actual: 1.4f}\n")
print(f"x_average ={mu_actual: 1.4f} ±{std_actual: 1.4f}\n")

In [ ]:
plotData(raw_dat=raw_groups, stack_dat=raw_ave, stack_std=raw_std, ave_dat=mu_actual, ave_std=std_actual)

# Using SST

In [ ]:
mu = (np.average(raw_0) + np.average(raw_1))/2
sst = va.calcSST([sample_len, sample_len],
                 [np.average(raw_0), np.average(raw_1)],
                 [np.std(raw_0), np.std(raw_1)],
                 mu)
# var_est_SST = sst / (2 * sample_len - 1)
var_est_SST = sst / (2 * sample_len)
std_est_SST = var_est_SST**0.5

print(f"mu ={mu: 1.4f}")
# print(f"sst ={sst: 1.4f}")
# print(f"var_est_SST ={var_est_SST: 1.4f}\n")
print(f"var_est_SST ={std_est_SST: 1.4f}\n")

# Using a direct method

In [ ]:
var_est_DM = ((np.var(raw_0) + np.var(raw_1)) / 2) + ((np.average(raw_0) - np.average(raw_1)) / 2)**2
std_est_DM = var_est_DM**0.5
print(f"mu ={mu: 1.4f}")
# print(f"var_est_DM ={var_est_DM: 1.4f}\n")
print(f"std_est_DM ={std_est_DM: 1.4f}\n")

In [ ]:
print('Two sets of samples')
print('---------------------------------')
print('\t\t     mu\t     var')
# print('Real \t'+str(mu_actual)+'\t'+str(var_actual))
print(f'Real\t\t{mu_actual: 1.4f}\t{std_actual: 1.4f}')
print(f'SST\t\t{mu: 1.4f}\t{std_est_SST: 1.4f}')
print(f'DirectMethod\t{mu: 1.4f}\t{std_est_DM: 1.4f}')

# Try with many sets ##################################

In [ ]:
num_new_sample = 20
print(f"sample_len = {sample_len}")

raw_groups = []
for ii in range(0, num_new_sample):
    raw_groups.append(np.random.normal(loc=np.random.random(1) * 10,
                                       scale=np.random.random(1) * 3 + 2,
                                       size=sample_len))
# print(f"raw_groups = {raw_groups}")

num_sample = len(raw_groups)
print(f"num_sample = {num_sample}\n")

print(f"size(raw_groups) = {len(raw_groups)} sets of {len(raw_groups[0])} samples")

raw_ts = np.concatenate(raw_groups)
# print(f"type(raw_ts) = {type(raw_ts)}")
print(f"raw_ts.size = {raw_ts.size}\n")

mu_actual = np.average(raw_ts)
var_actual = np.var(raw_ts)
std_actual = np.std(raw_ts)
print(f"mu_actual = {mu_actual: 1.4f}")
# print(f"var_actual = {var_actual: 1.4f}\n")
print(f"std_actual = {std_actual: 1.4f}\n")

mu_est_SST = 0
mu_ind_iv = []
var_ind_iv = []
std_ind_iv = []
sample_weight = []
for a in raw_groups:
    mu_est_SST += np.average(a) / num_sample
    # mu_est_SST += np.average(a) / num_sample
    mu_ind_iv.append(np.average(a))
    var_ind_iv.append(np.var(a))
    std_ind_iv.append(np.std(a))
    # sample_weight.append(sample_len)
    sample_weight.append(1)

sst = va.calcSST(sample_weight, mu_ind_iv, std_ind_iv, mu_est_SST)
var_est_SST_long = sst / np.sum(sample_weight)

var_est_SST = va.calcVarSST(sample_weight, mu_ind_iv, std_ind_iv, mu_est_SST)

std_est_SST = var_est_SST**0.5

print(f"mu_est_SST = {mu_est_SST: 1.4f}")
print(f"var_est_SST_long = {var_est_SST_long: 1.4f}\n")
print(f"var_est_SST = {var_est_SST: 1.4f}\n")
print(f"std_est_SST = {std_est_SST: 1.4f}\n")

print(f'\n{num_sample} sets of samples')
print(f'---------------------------------')
print(f'\t     mu\t     var\t    std')
print(f'Real\t{mu_actual: 1.4f}\t{var_actual: 1.4f}\t{std_actual: 1.4f}')
print(f'SST\t{mu_est_SST: 1.4f}\t{var_est_SST: 1.4f}\t{std_est_SST: 1.4f}')

In [ ]:
plotData(raw_dat=raw_groups, stack_dat=mu_ind_iv, stack_std=std_ind_iv, ave_dat=mu_actual, ave_std=std_est_SST)